# Aesthetic Image Scoring — Experiments

All logic lives in `src/`. This notebook only runs experiments and collects
results, so re-running it top to bottom is cheap and always reproduces the
table at the bottom.

Requires a GPU runtime: `Runtime -> Change runtime type -> T4 GPU`.

## Setup

On Colab, clone the repository and install dependencies. Locally, skip this
cell — just make sure the working directory is the project root.

In [ ]:
import os

if not os.path.exists('src'):
    # Colab: replace with your own repository URL
    !git clone -q https://github.com/YOUR_USERNAME/aesthetic-scoring.git
    %cd aesthetic-scoring
    !pip install -q -r requirements.txt

import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## Feature extraction

Run once per backbone. Each call writes `{backbone}_train.npy` and
`{backbone}_test.npy` into `features/` and skips the work entirely if those
files already exist.

Point `--images` at the unzipped AADB directory. On Colab, mounting Drive and
unzipping from there survives runtime restarts; extracting into `/content` does
not.

Expect roughly two minutes per backbone on a T4 for the ~9.5k images.

In [ ]:
IMAGES = 'data/datasetImages_warp256'   # adjust to your unzipped archive

!python -m src.extract_features --backbone clip     --images {IMAGES}
!python -m src.extract_features --backbone resnet50 --images {IMAGES}

## Experiments

`run_experiment` loads the cached features, holds out 10% of training data for
validation, fits the head with early stopping on validation SRCC, and evaluates
on the test set once.

Each configuration below isolates one variable, so the differences between rows
are interpretable rather than confounded.

In [ ]:
from src.train import run_experiment

results = {}

results['CLIP ViT-B/32'] = run_experiment(['clip'], 'features')

### CNN branch

ResNet-50 features alone, for comparison. Expect this to trail CLIP: ImageNet
pretraining optimises for object identity, which overlaps with aesthetic
judgement but is not the same thing.

In [ ]:
results['ResNet-50'] = run_experiment(['resnet50'], 'features')

### Late fusion

Both views together. Whether fusion helps is an empirical question — if the two
backbones encode largely the same information about aesthetics, the extra
parameters buy nothing and may cost a little through overfitting.

In [ ]:
results['CLIP + ResNet-50'] = run_experiment(['clip', 'resnet50'], 'features')

### Ranking loss

The metrics measure ordering while MSE measures absolute error. Adding a
pairwise margin term optimises the ranking objective directly.

In [ ]:
results['CLIP + rank loss'] = run_experiment(['clip'], 'features', loss_name='mse+rank')
results['Fusion + rank loss'] = run_experiment(
    ['clip', 'resnet50'], 'features', loss_name='mse+rank'
)

## Ablation table

In [ ]:
import pandas as pd

df = (pd.DataFrame(results).T[['SRCC', 'KRCC', 'MSE', 'best_epoch']]
        .sort_values('SRCC', ascending=False)
        .round(4))
df

Reference point: the original AADB paper (Kong et al., ECCV 2016) reports SRCC
0.678 on this test set.

Read any improvement over it as the contribution of large-scale pretraining
rather than of a better head — CLIP's 400M image-text pairs were not available
in 2016, and the head here is deliberately minimal.

Validation and test scores should sit close together. A large gap in either
direction is a symptom rather than a result, and usually means a stale model was
reused across runs or the caches fell out of sync.

## Robustness check

A single seed can flatter or punish a configuration by chance, especially when
two rows differ by a point or two of SRCC. Repeating the best configurations
across seeds shows whether a gap is real.

In [ ]:
import numpy as np

for name, feats, loss in [('CLIP', ['clip'], 'mse'),
                          ('Fusion + rank', ['clip', 'resnet50'], 'mse+rank')]:
    scores = [run_experiment(feats, 'features', seed=s, loss_name=loss)['SRCC']
              for s in range(3)]
    print(f"{name:16s} SRCC {np.mean(scores):.4f} +/- {np.std(scores):.4f}")